<a href="https://colab.research.google.com/github/samuelaojih/Google-Colab/blob/main/Irrigation_Suitability_MCDA_Katagun.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Irrigation Suitability MCDA — Katagun-K.Gana Catchment

Multi-Criteria Decision Analysis (AHP-weighted overlay) for irrigation site suitability, using 7 criteria:
**AWC, Slope, Road Access, Rainfall, Productivity, Population, Drainage Proximity.**

Study area: the `Katagun-K.Gana` feature of `projects/ee-samuelachonuojih/assets/Catchment`.
Soil data (AWC source): `projects/ee-samuelcool28/assets/ACRESAL_Soil_Data`.


In [ ]:
pip install earthengine-api geemap

In [ ]:
import ee
import geemap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
ee.Authenticate()
ee.Initialize(
    project='ee-samuelcool28',
    opt_url='https://earthengine-highvolume.googleapis.com'
)


## 1. Study Area — Katagun-K.Gana Catchment

In [ ]:
CATCHMENT_ASSET = 'projects/ee-samuelachonuojih/assets/Catchment'
SOIL_ASSET = 'projects/ee-samuelcool28/assets/ACRESAL_Soil_Data'

catchments = ee.FeatureCollection(CATCHMENT_ASSET)
katagunCollection = catchments.filter(ee.Filter.eq('NAME', 'Katagun-K.Gana'))
katagunFeature = katagunCollection.first()
katagun = katagunFeature.geometry()

area_km2 = katagun.area(1).divide(1e6)
print('Selected catchment:', katagunFeature.get('NAME').getInfo())
print('Area (km2):', area_km2.getInfo())


## 2. Inspect the ACRESAL Soil Data asset

The AWC (Available Water Capacity) band/property name isn't known in advance, so this cell
prints the asset type and its bands so you can confirm the correct name before proceeding.


In [ ]:
soilAssetInfo = ee.data.getAsset(SOIL_ASSET)
print('Asset type:', soilAssetInfo['type'])

if soilAssetInfo['type'] == 'IMAGE_COLLECTION':
    soilCollection = ee.ImageCollection(SOIL_ASSET)
    print('Number of images:', soilCollection.size().getInfo())
    print('Bands:', soilCollection.first().bandNames().getInfo())
    soilImage = soilCollection.mosaic()
elif soilAssetInfo['type'] == 'IMAGE':
    soilImage = ee.Image(SOIL_ASSET)
    print('Bands:', soilImage.bandNames().getInfo())
else:
    print('Unexpected asset type, inspect manually:', soilAssetInfo)
    soilImage = None


In [ ]:
# <-- EDIT this to match the exact band name printed above for Available Water Capacity
AWC_BAND_NAME = 'AWC'

awc = soilImage.select(AWC_BAND_NAME).clip(katagun).rename('AWC')


## 3. Build the 7 Criteria Layers

`ANALYSIS_SCALE` is the common working resolution (meters) used for normalization statistics,
the weighted overlay, zonal statistics, and export.


In [ ]:
ANALYSIS_SCALE = 30

def get_min_max(image, geometry, scale=ANALYSIS_SCALE):
    """Compute the min/max of a single-band image within geometry (server-side ee.Numbers)."""
    band = image.bandNames().get(0)
    stats = image.reduceRegion(
        reducer=ee.Reducer.minMax(),
        geometry=geometry,
        scale=scale,
        bestEffort=True,
        maxPixels=1e13,
        tileScale=4
    )
    return ee.Number(stats.get(ee.String(band).cat('_min'))), ee.Number(stats.get(ee.String(band).cat('_max')))

def normalize(image, geometry, invert=False, scale=ANALYSIS_SCALE):
    """Min-max normalize a single-band image to [0, 1] within geometry.
    invert=True treats the criterion as a 'cost' (lower raw value = more suitable).
    """
    min_val, max_val = get_min_max(image, geometry, scale)
    norm = image.subtract(min_val).divide(max_val.subtract(min_val)).max(0).min(1)
    return norm.multiply(-1).add(1) if invert else norm


In [ ]:
# ---------------------- SLOPE (from SRTM DEM) ------------------------ #
dem = ee.Image('USGS/SRTMGL1_003').clip(katagun)
slope = ee.Terrain.slope(dem).rename('Slope')


In [ ]:
# ---------------------- ROAD ACCESS (GRIP4 roads) --------------------- #
ROADS_ASSET = 'projects/sat-io/open-datasets/GRIP4/Africa'
roads = ee.FeatureCollection(ROADS_ASSET).filterBounds(katagun.buffer(20000))

print('Road feature property names (use to filter by road class if desired):')
print(roads.first().propertyNames().getInfo())

roadDistance = roads.distance(searchRadius=50000, maxError=100).clip(katagun).rename('RoadAccess')


In [ ]:
# ---------------------- RAINFALL (CHIRPS, mean annual) ----------------- #
startyear = 2015
endyear = 2024
years = list(range(startyear, endyear + 1))

chirps = ee.ImageCollection('UCSB-CHG/CHIRPS/PENTAD')
rainImages = [chirps.filter(ee.Filter.calendarRange(y, y, 'year')).sum().set('year', y) for y in years]
rainfall = ee.ImageCollection(rainImages).mean().clip(katagun).rename('Rainfall')


In [ ]:
# ---------------------- PRODUCTIVITY (MODIS annual NPP) ---------------- #
npp = ee.ImageCollection('MODIS/061/MOD17A3HGF') \
    .filterDate(f'{endyear}-01-01', f'{endyear + 1}-01-01') \
    .select('Npp').mean().multiply(0.0001).clip(katagun).rename('Productivity')


In [ ]:
# ---------------------- POPULATION (WorldPop) --------------------------- #
POP_YEAR = 2020
population = ee.ImageCollection('WorldPop/GP/100m/pop') \
    .filter(ee.Filter.eq('country', 'NGA')) \
    .filter(ee.Filter.eq('year', POP_YEAR)) \
    .mosaic().clip(katagun).rename('Population')


In [ ]:
# ---------------------- DRAINAGE PROXIMITY (HydroSHEDS flow accumulation) - #
flowAccum = ee.Image('WWF/HydroSHEDS/15ACC').select('b1').clip(katagun)

streamThreshold = ee.Number(flowAccum.reduceRegion(
    reducer=ee.Reducer.percentile([95]),
    geometry=katagun, scale=463, bestEffort=True, maxPixels=1e13, tileScale=4
).get('b1'))

streamMask = flowAccum.gte(streamThreshold)
drainageProximity = streamMask.fastDistanceTransform(256).sqrt().clip(katagun).rename('DrainageProx')


## 4. Normalize All Criteria (0–1, benefit vs. cost)

In [ ]:
# invert=True marks a 'cost' criterion (lower raw value = more suitable).
# Review these assumptions against your own literature/expert judgement and flip as needed.
CRITERIA_CONFIG = {
    'AWC':          {'image': awc,             'invert': False},  # higher water-holding capacity -> more suitable
    'Slope':        {'image': slope,           'invert': True},   # steeper slope -> less suitable
    'RoadAccess':   {'image': roadDistance,    'invert': True},   # farther from roads -> less suitable
    'Rainfall':     {'image': rainfall,        'invert': False},  # higher rainfall -> more suitable
    'Productivity': {'image': npp,             'invert': False},  # higher NPP -> more suitable
    'Population':   {'image': population,      'invert': False},  # higher density -> more suitable (labour/market access)
    'DrainageProx': {'image': drainageProximity, 'invert': True}, # farther from drainage -> less suitable
}

normalized = {}
for name, cfg in CRITERIA_CONFIG.items():
    normalized[name] = normalize(cfg['image'], katagun, invert=cfg['invert']).rename(name)
    print(f'Normalized {name}')


## 5. AHP Pairwise Comparison & Criteria Weights

Weights are derived with the eigenvector method (Saaty, 1980) from a pairwise comparison
matrix built from a ranked order of importance, then checked for consistency (CR < 0.10).
Edit `IMPORTANCE_RANK` (1 = most important) to reflect your own expert judgement.


In [ ]:
CRITERIA = ['AWC', 'Slope', 'RoadAccess', 'Rainfall', 'Productivity', 'Population', 'DrainageProx']

IMPORTANCE_RANK = {
    'AWC': 1, 'Slope': 2, 'DrainageProx': 3, 'Rainfall': 4,
    'Productivity': 5, 'RoadAccess': 6, 'Population': 7
}

SAATY_SCALE_BY_GAP = {0: 1, 1: 2, 2: 3, 3: 4, 4: 5, 5: 7, 6: 9}

n = len(CRITERIA)
A = np.ones((n, n))
for i in range(n):
    for j in range(n):
        if i == j:
            continue
        ri, rj = IMPORTANCE_RANK[CRITERIA[i]], IMPORTANCE_RANK[CRITERIA[j]]
        s = SAATY_SCALE_BY_GAP[abs(ri - rj)]
        A[i, j] = s if ri < rj else 1 / s

eigvals, eigvecs = np.linalg.eig(A)
maxIdx = np.argmax(eigvals.real)
lamMax = eigvals[maxIdx].real
weightsVec = eigvecs[:, maxIdx].real
weightsVec = weightsVec / weightsVec.sum()

RI_TABLE = {1: 0, 2: 0, 3: 0.58, 4: 0.9, 5: 1.12, 6: 1.24, 7: 1.32, 8: 1.41, 9: 1.45, 10: 1.49}
CI = (lamMax - n) / (n - 1)
CR = CI / RI_TABLE[n]

WEIGHTS = dict(zip(CRITERIA, weightsVec))

print(pd.DataFrame({'Criterion': CRITERIA, 'Weight': [WEIGHTS[c] for c in CRITERIA]})
      .sort_values('Weight', ascending=False).to_string(index=False))
print(f'\nConsistency Ratio (CR): {CR:.4f}', '-> acceptable (< 0.10)' if CR < 0.10 else '-> REVISE the pairwise judgements')


## 6. Weighted Overlay — Irrigation Suitability Index

In [ ]:
irrigationSuitability = ee.Image(0)
for name in CRITERIA:
    irrigationSuitability = irrigationSuitability.add(normalized[name].multiply(WEIGHTS[name]))

irrigationSuitability = irrigationSuitability.clip(katagun).rename('suitability')


## 7. Reclassify into Suitability Classes

In [ ]:
# 1 = Not Suitable, 2 = Marginally Suitable, 3 = Moderately Suitable, 4 = Suitable, 5 = Highly Suitable
suitabilityClass = ee.Image(1) \
    .where(irrigationSuitability.gte(0.2), 2) \
    .where(irrigationSuitability.gte(0.4), 3) \
    .where(irrigationSuitability.gte(0.6), 4) \
    .where(irrigationSuitability.gte(0.8), 5) \
    .clip(katagun).rename('suitability_class')


## 8. Visualization

In [ ]:
Map = geemap.Map()
Map.centerObject(katagun, 10)

Map.addLayer(ee.Image().paint(katagunCollection, 0, 2), {'palette': ['black']}, 'Katagun-K.Gana Boundary')

Map.addLayer(irrigationSuitability, {
    'min': 0, 'max': 1,
    'palette': ['#d73027', '#fc8d59', '#fee08b', '#d9ef8b', '#91cf60', '#1a9850']
}, 'Irrigation Suitability Index (0-1)')

classPalette = ['#d73027', '#fc8d59', '#fee08b', '#91cf60', '#1a9850']
Map.addLayer(suitabilityClass, {'min': 1, 'max': 5, 'palette': classPalette}, 'Suitability Classes', False)

Map.add_legend(title='Irrigation Suitability', legend_dict={
    '1 - Not Suitable': '#d73027',
    '2 - Marginally Suitable': '#fc8d59',
    '3 - Moderately Suitable': '#fee08b',
    '4 - Suitable': '#91cf60',
    '5 - Highly Suitable': '#1a9850',
})

Map.addLayerControl()
Map


## 9. Zonal Area Statistics per Suitability Class

In [ ]:
CLASS_NAMES = {1: 'Not Suitable', 2: 'Marginally Suitable', 3: 'Moderately Suitable',
               4: 'Suitable', 5: 'Highly Suitable'}

areaByClass = ee.Image.pixelArea().divide(1e6).addBands(suitabilityClass).reduceRegion(
    reducer=ee.Reducer.sum().group(groupField=1, groupName='class'),
    geometry=katagun,
    scale=ANALYSIS_SCALE,
    bestEffort=True,
    maxPixels=1e13,
    tileScale=4
).get('groups').getInfo()

statsDf = pd.DataFrame(areaByClass).rename(columns={'class': 'class_id', 'sum': 'area_km2'})
statsDf['Class'] = statsDf['class_id'].map(CLASS_NAMES)
statsDf['area_pct'] = 100 * statsDf['area_km2'] / statsDf['area_km2'].sum()
statsDf = statsDf[['Class', 'area_km2', 'area_pct']].sort_values('area_km2', ascending=False)
print(statsDf.to_string(index=False))

statsDf.plot(x='Class', y='area_km2', kind='bar', legend=False,
             color=[classPalette[c - 1] for c in sorted(CLASS_NAMES) if CLASS_NAMES[c] in statsDf['Class'].values])
plt.ylabel('Area (km2)')
plt.title('Katagun-K.Gana Irrigation Suitability — Area per Class')
plt.tight_layout()
plt.show()


## 10. Export

In [ ]:
exportIndexTask = ee.batch.Export.image.toDrive(
    image=irrigationSuitability.toFloat(),
    description='Katagun_Irrigation_Suitability_Index',
    folder='GEE_Exports',
    fileNamePrefix='Katagun_Irrigation_Suitability_Index',
    region=katagun,
    scale=ANALYSIS_SCALE,
    maxPixels=1e13
)
exportIndexTask.start()

exportClassTask = ee.batch.Export.image.toDrive(
    image=suitabilityClass.toByte(),
    description='Katagun_Irrigation_Suitability_Classes',
    folder='GEE_Exports',
    fileNamePrefix='Katagun_Irrigation_Suitability_Classes',
    region=katagun,
    scale=ANALYSIS_SCALE,
    maxPixels=1e13
)
exportClassTask.start()

print('Export tasks started - check the "Tasks" tab at https://code.earthengine.google.com/tasks')
